# Tutorial Notebook: Fairness Monitoring Warm-Up

This tutorial is a compact rehearsal for Project 4. You will build a credit-risk style model, audit group fairness, try one mitigation, and simulate simple production monitoring.

The main project uses a substantial HMDA mortgage slice. This warmup uses the saved UCI Default of Credit Card Clients file at `data/default_credit_card_clients.csv`, with OpenML/offline fallback logic only as a backup.


## 1) Responsible-AI Framing

Scenario: a lender uses a model to prioritize applications for enhanced review.

Warmup decision: flag high-risk credit accounts for manual review.

Audit question: does the model perform similarly across demographic groups, and would it require controls before deployment?

Interpretation rule: a fairness metric is not a legal conclusion. It is an audit signal that requires context, sample sizes, uncertainty, and investigation.


## 2) Setup


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_classification
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, f1_score, precision_score, recall_score,
    roc_auc_score, brier_score_loss
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 4146
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("data")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)


## 3) Load Warmup Data

Preferred source: saved UCI Default of Credit Card Clients file.

Expected local file:
- `data/default_credit_card_clients.csv`

If that file is missing, the cell also checks `data/UCI_Credit_Card.csv`, then tries OpenML. If network access is unavailable, the notebook creates a deterministic credit-like fallback dataset so the tutorial can still demonstrate the mechanics.


In [ ]:
def make_offline_credit_like_data(n=6000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    X, y = make_classification(
        n_samples=n, n_features=8, n_informative=5, n_redundant=1,
        weights=[0.78, 0.22], class_sep=0.9, random_state=seed
    )
    df = pd.DataFrame(X, columns=[f"risk_signal_{i}" for i in range(8)])
    sex = rng.choice([1, 2], size=n, p=[0.42, 0.58])
    education = rng.choice([1, 2, 3, 4], size=n, p=[0.28, 0.39, 0.25, 0.08])
    age = np.clip((rng.normal(37, 10, n)).round(), 21, 75).astype(int)
    limit_bal = np.exp(rng.normal(11.6, 0.75, n)).round(-2)

    y = y.copy()
    risk_bump = ((sex == 1) & (education >= 3) & (rng.random(n) < 0.08))
    y[risk_bump] = 1

    df["LIMIT_BAL"] = limit_bal
    df["SEX"] = sex
    df["EDUCATION"] = education
    df["AGE"] = age
    df["default_payment_next_month"] = y
    df["source_note"] = "offline synthetic fallback"
    return df


def load_warmup_data():
    candidates = [DATA_DIR / "default_credit_card_clients.csv", DATA_DIR / "UCI_Credit_Card.csv"]
    for path in candidates:
        if path.exists():
            df = pd.read_csv(path)
            return df, f"local file: {path}"

    try:
        from sklearn.datasets import fetch_openml
        raw = fetch_openml("default-of-credit-card-clients", version=1, as_frame=True)
        df = raw.frame.copy()
        return df, "OpenML: default-of-credit-card-clients"
    except Exception as exc:
        df = make_offline_credit_like_data()
        return df, f"offline fallback because real-data load failed: {type(exc).__name__}"

raw_df, data_source = load_warmup_data()
print(data_source)
print(raw_df.shape)
raw_df.head()


## 4) Standardize Target and Sensitive Attributes

For the warmup only, we use `SEX`, `EDUCATION`, and age bands as audit groups. In Project 4, sensitive attributes will come from HMDA fields such as race, ethnicity, sex, and age.


In [ ]:
df = raw_df.copy()
df.columns = [str(c).strip().replace(" ", "_") for c in df.columns]

# OpenML may expose the UCI columns as x1..x23 plus y. Normalize those aliases.
openml_aliases = {"x2": "SEX", "x3": "EDUCATION", "x5": "AGE", "y": "default_payment_next_month"}
for old, new in openml_aliases.items():
    if old in df.columns and new not in df.columns:
        df[new] = df[old]

target_candidates = ["default_payment_next_month", "default.payment.next.month", "Y", "y", "target", "default"]
target_col = next((c for c in target_candidates if c in df.columns), None)
if target_col is None and "x24" in df.columns:
    target_col = "x24"
if target_col is None:
    raise ValueError(f"Could not find target column. Available columns: {list(df.columns)[:30]}")

df["target_default"] = pd.to_numeric(df[target_col], errors="coerce").astype(int)
if "SEX" not in df.columns:
    raise ValueError("Warmup expects a SEX column or the built-in fallback data.")

df["sex_group"] = df["SEX"].map({1: "sex_1", 2: "sex_2"}).fillna("unknown")
if "AGE" in df.columns:
    df["age_group"] = pd.cut(pd.to_numeric(df["AGE"], errors="coerce"), bins=[0, 29, 44, 59, 120], labels=["under_30", "30_44", "45_59", "60_plus"])
else:
    df["age_group"] = "unknown"
if "EDUCATION" in df.columns:
    df["education_group"] = "edu_" + df["EDUCATION"].astype(str)
else:
    df["education_group"] = "unknown"

print(df["target_default"].value_counts(normalize=True).rename("rate"))
df[["target_default", "sex_group", "age_group", "education_group"]].head()


## 5) Train / Validation / Test Split

Train fits the model. Validation chooses thresholds and mitigation settings. Test is held for a final snapshot.


In [ ]:
target_like_cols = {target_col, "target_default", "source_note"}
for possible_target in ["default_payment_next_month", "default.payment.next.month", "Y", "y", "target", "default", "x24"]:
    if possible_target in df.columns:
        target_like_cols.add(possible_target)
feature_cols = [c for c in df.columns if c not in target_like_cols]
X = df[feature_cols].copy()
y = df["target_default"].copy()

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=RANDOM_SEED)

def split_summary(name, y_part):
    return {"split": name, "n": len(y_part), "default_rate": float(y_part.mean())}

pd.DataFrame([split_summary("train", y_train), split_summary("validation", y_val), split_summary("test", y_test)])


## 6) Baseline Models

We train one interpretable model and one higher-capacity model. The point is not leaderboard tuning; it is to create a model worth auditing.


In [ ]:
numeric_features = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
categorical_features = [c for c in X_train.columns if c not in numeric_features]

linear_preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])
tree_preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])

models = {
    "logistic_regression": Pipeline([("prep", linear_preprocess), ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED))]),
    "hist_gradient_boosting": Pipeline([("prep", tree_preprocess), ("model", HistGradientBoostingClassifier(max_iter=120, learning_rate=0.06, random_state=RANDOM_SEED))]),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_val)[:, 1]
    pred = (proba >= 0.50).astype(int)
    rows.append({
        "model": name,
        "roc_auc": roc_auc_score(y_val, proba),
        "brier": brier_score_loss(y_val, proba),
        "precision@0.50": precision_score(y_val, pred, zero_division=0),
        "recall@0.50": recall_score(y_val, pred, zero_division=0),
        "f1@0.50": f1_score(y_val, pred, zero_division=0),
    })
model_report = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
model_report


## 7) Threshold Policy

A threshold is a policy choice. Here we choose the validation threshold that minimizes a simple review-cost model.


In [ ]:
best_name = model_report.iloc[0]["model"]
best_model = models[best_name]
val_proba = best_model.predict_proba(X_val)[:, 1]

review_cost = 1.0
miss_cost = 8.0
max_flag_rate = 0.35

threshold_rows = []
for threshold in np.linspace(0.05, 0.85, 33):
    pred = (val_proba >= threshold).astype(int)
    flagged = pred.mean()
    fp = int(((pred == 1) & (y_val.values == 0)).sum())
    fn = int(((pred == 0) & (y_val.values == 1)).sum())
    total_cost = review_cost * int(pred.sum()) + miss_cost * fn
    threshold_rows.append({
        "threshold": threshold,
        "flag_rate": flagged,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "fp": fp,
        "fn": fn,
        "cost": total_cost,
        "within_capacity": flagged <= max_flag_rate,
    })
threshold_table = pd.DataFrame(threshold_rows)
feasible = threshold_table[threshold_table["within_capacity"]]
chosen = feasible.sort_values("cost").iloc[0] if len(feasible) else threshold_table.sort_values("cost").iloc[0]
print(f"Selected model: {best_name}")
print(chosen.to_dict())
threshold_table.sort_values("cost").head(10)


## 8) Mini Tutorial: Fairness Is Not One Metric

Fairness auditing uses multiple definitions because different definitions protect against different kinds of harm. They can also conflict, especially when historical outcome rates differ across groups.

### Group Fairness Families

Group fairness compares model behavior across protected or vulnerable groups. Common definitions include:

- **Demographic parity / statistical parity:** groups should receive positive decisions at similar rates. This focuses on outcome allocation, regardless of the historical label.
- **Equal opportunity:** groups should have similar true-positive rates. This focuses on whether qualified or positive-label cases are recognized at similar rates.
- **Equalized odds:** groups should have similar true-positive and false-positive rates. This focuses on both kinds of classification error.
- **Predictive parity:** positive predictions should mean similar things across groups, often measured with precision or positive predictive value.
- **Calibration within groups:** predicted probabilities should match observed outcome rates within each group.
- **Worst-group performance:** the system should not hide a badly served group behind strong average performance.

No single group metric is automatically the right one. In high-stakes lending, you must explain which errors are most harmful, which metric aligns with the decision, and where definitions disagree.

### Individual Fairness Families

Individual fairness asks whether similar people are treated similarly. This is harder because it depends on how similarity is defined. Common versions include:

- **Similarity-based consistency:** applicants similar on decision-relevant, non-sensitive features should receive similar scores or decisions.
- **Counterfactual fairness:** a prediction should not change only because a protected attribute changes, holding the appropriate causal background fixed. This requires causal assumptions, not just a table of features.
- **Nearest-neighbor inconsistency:** matched or near-matched applicants should not have large unexplained score gaps. This is a practical diagnostic, not a full proof.
- **Individual recourse fairness:** people with similar profiles should have similarly feasible paths to improve an outcome.

In this warmup, the group section computes selection rates, TPR/FPR/FNR, precision, and mean score by group. The individual section uses a nearest-neighbor consistency diagnostic. Project 4 will ask for more explicit justification of which fairness definitions matter for mortgage decision support.


## 9) Group Fairness Metrics

This function reports sample size, selection rate, error rates, and predictive value by group. In a real audit, small groups should be interpreted cautiously.


In [ ]:
def group_metric_table(X_part, y_true, proba, group_col, threshold):
    pred = (proba >= threshold).astype(int)
    work = pd.DataFrame({"y": np.asarray(y_true), "pred": pred, "score": proba, "group": X_part[group_col].astype(str).values})
    rows = []
    for group, g in work.groupby("group"):
        tn, fp, fn, tp = confusion_matrix(g["y"], g["pred"], labels=[0, 1]).ravel()
        n = len(g)
        selection_rate = g["pred"].mean()
        rows.append({
            "group_col": group_col,
            "group": group,
            "n": n,
            "base_rate": g["y"].mean(),
            "selection_rate": selection_rate,
            "selection_rate_ci_halfwidth": 1.96 * math.sqrt(max(selection_rate * (1 - selection_rate), 0) / n),
            "tpr": tp / (tp + fn) if (tp + fn) else np.nan,
            "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
            "fnr": fn / (tp + fn) if (tp + fn) else np.nan,
            "precision": tp / (tp + fp) if (tp + fp) else np.nan,
            "mean_score": g["score"].mean(),
        })
    return pd.DataFrame(rows).sort_values("n", ascending=False)

fairness_tables = []
for group_col in ["sex_group", "age_group", "education_group"]:
    if group_col in X_val.columns:
        fairness_tables.append(group_metric_table(X_val, y_val, val_proba, group_col, float(chosen["threshold"])))
fairness_report = pd.concat(fairness_tables, ignore_index=True)
fairness_report


## 10) Simple Mitigation: Remove Sensitive Columns

This is deliberately modest. Removing protected attributes is not proof of fairness, because other features can act as proxies. It is useful as a first comparison point.


In [ ]:
sensitive_cols = [c for c in ["SEX", "sex_group", "AGE", "age_group", "EDUCATION", "education_group"] if c in X_train.columns]
X_train_m = X_train.drop(columns=sensitive_cols)
X_val_m = X_val.drop(columns=sensitive_cols)

num_m = [c for c in X_train_m.columns if pd.api.types.is_numeric_dtype(X_train_m[c])]
cat_m = [c for c in X_train_m.columns if c not in num_m]
prep_m = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_m),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_m),
])
mitigated_model = Pipeline([("prep", prep_m), ("model", HistGradientBoostingClassifier(max_iter=120, learning_rate=0.06, random_state=RANDOM_SEED))])
mitigated_model.fit(X_train_m, y_train)
val_proba_m = mitigated_model.predict_proba(X_val_m)[:, 1]

comparison = pd.DataFrame([
    {"system": "baseline", "roc_auc": roc_auc_score(y_val, val_proba), "brier": brier_score_loss(y_val, val_proba)},
    {"system": "sensitive_columns_removed", "roc_auc": roc_auc_score(y_val, val_proba_m), "brier": brier_score_loss(y_val, val_proba_m)},
])
comparison


## 11) Before / After Fairness Snapshot


In [ ]:
baseline_sex = group_metric_table(X_val, y_val, val_proba, "sex_group", float(chosen["threshold"]))
mitigated_sex = group_metric_table(X_val, y_val, val_proba_m, "sex_group", float(chosen["threshold"]))
sex_compare = baseline_sex[["group", "n", "selection_rate", "tpr", "fpr", "precision"]].merge(
    mitigated_sex[["group", "selection_rate", "tpr", "fpr", "precision"]],
    on="group", suffixes=("_baseline", "_mitigated")
)
sex_compare


## 12) Individual Fairness Diagnostic

A simple check: among nearest neighbors using non-sensitive numeric features, how often do similar applicants receive substantially different scores? This is a diagnostic, not a full fairness proof.


In [ ]:
nn_features = [c for c in numeric_features if c not in ["SEX", "AGE", "EDUCATION"]]
if len(nn_features) >= 2:
    nn_X = X_val[nn_features].copy()
    nn_X = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(nn_X), columns=nn_features)
    nn_X = pd.DataFrame(StandardScaler().fit_transform(nn_X), columns=nn_features)
    nn = NearestNeighbors(n_neighbors=2).fit(nn_X)
    distances, indices = nn.kneighbors(nn_X)
    nearest_idx = indices[:, 1]
    score_diff = np.abs(val_proba - val_proba[nearest_idx])
    individual_report = pd.DataFrame({
        "mean_nearest_neighbor_distance": [distances[:, 1].mean()],
        "mean_score_difference": [score_diff.mean()],
        "large_difference_rate_gt_0_20": [(score_diff > 0.20).mean()],
    })
else:
    individual_report = pd.DataFrame({"note": ["Not enough numeric non-sensitive features for nearest-neighbor diagnostic."]})
individual_report


## 13) Monitoring Simulation

We split validation data into pseudo-production batches and track model and fairness signals over time. Project 4 will do this with real HMDA time fields.


In [ ]:
monitor_df = X_val.copy().reset_index(drop=True)
monitor_df["y"] = y_val.reset_index(drop=True)
monitor_df["score"] = val_proba
monitor_df["pred"] = (val_proba >= float(chosen["threshold"])).astype(int)
monitor_df["batch"] = pd.qcut(np.arange(len(monitor_df)), q=6, labels=[f"batch_{i}" for i in range(1, 7)])

monitor_rows = []
for batch, g in monitor_df.groupby("batch", observed=True):
    for sex_group, sg in g.groupby("sex_group"):
        monitor_rows.append({
            "batch": batch,
            "sex_group": sex_group,
            "n": len(sg),
            "default_rate": sg["y"].mean(),
            "mean_score": sg["score"].mean(),
            "selection_rate": sg["pred"].mean(),
            "recall": recall_score(sg["y"], sg["pred"], zero_division=0),
        })
monitoring_table = pd.DataFrame(monitor_rows)
monitoring_table


## 14) Alert Rule Example

A simple alert fires when the selection-rate gap between sex groups exceeds 0.10 in a batch with enough observations. Alerts initiate investigation; they do not prove discrimination by themselves.


In [ ]:
alert_rows = []
for batch, g in monitoring_table.groupby("batch", observed=True):
    if (g["n"] >= 50).sum() >= 2:
        gap = g["selection_rate"].max() - g["selection_rate"].min()
        alert_rows.append({
            "batch": batch,
            "selection_rate_gap": gap,
            "alert": gap > 0.10,
            "investigation_note": "Check group composition, score drift, and error rates before making a deployment decision." if gap > 0.10 else "No alert."
        })
alerts = pd.DataFrame(alert_rows)
alerts


## 15) Final Test Snapshot

Use the locked validation threshold once on the test set.


In [ ]:
test_proba = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= float(chosen["threshold"])).astype(int)
test_report = {
    "model": best_name,
    "threshold": float(chosen["threshold"]),
    "roc_auc": roc_auc_score(y_test, test_proba),
    "brier": brier_score_loss(y_test, test_proba),
    "precision": precision_score(y_test, test_pred, zero_division=0),
    "recall": recall_score(y_test, test_pred, zero_division=0),
    "f1": f1_score(y_test, test_pred, zero_division=0),
}
test_report


## 16) Artifact Export


In [ ]:
model_report.to_csv(ARTIFACT_DIR / "warmup_model_report.csv", index=False)
threshold_table.to_csv(ARTIFACT_DIR / "warmup_threshold_table.csv", index=False)
fairness_report.to_csv(ARTIFACT_DIR / "warmup_fairness_report.csv", index=False)
monitoring_table.to_csv(ARTIFACT_DIR / "warmup_monitoring_table.csv", index=False)
alerts.to_csv(ARTIFACT_DIR / "warmup_alerts.csv", index=False)
with open(ARTIFACT_DIR / "warmup_test_report.json", "w") as f:
    json.dump(test_report, f, indent=2)
print(f"Saved warmup artifacts to {ARTIFACT_DIR.resolve()}")


## 17) What To Carry Into Project 4

1. Historical labels are not neutral ground truth.
2. Overall ROC-AUC can hide group-specific harm.
3. A mitigation can improve one metric while worsening another.
4. Monitoring needs sample sizes, thresholds, and investigation steps.
5. Deployment recommendations should be evidence-based: deploy, deploy with controls, remediate, or reject.
